In [1]:
!pip install openai

In [4]:
# ==============================
# 🩺 FREE General Health Chatbot (Hugging Face - Colab)
# ==============================

# Install required libraries
!pip install -q transformers accelerate torch

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ------------------------------
# 1️⃣ Load Free Model
# ------------------------------
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

print("Loading model... please wait (first time takes 1-2 minutes)")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully ✅")

# ------------------------------
# 2️⃣ Prompt Engineering
# ------------------------------
SYSTEM_PROMPT = """
You are a helpful and friendly medical assistant.

Rules:
- Provide general health information only.
- Do NOT diagnose diseases.
- Do NOT prescribe medication.
- Use simple and clear language.
- If symptoms are serious, advise seeing a doctor.
- Always end your response with:
"This information is for educational purposes only and not medical advice."
"""

# ------------------------------
# 3️⃣ Safety Filter
# ------------------------------
def safety_filter(user_input):
    dangerous_keywords = [
        "suicide", "kill myself", "overdose",
        "self harm", "how many pills",
        "dosage", "prescribe", "end my life"
    ]
    
    for word in dangerous_keywords:
        if word in user_input.lower():
            return True
    return False

# ------------------------------
# 4️⃣ Chat Function
# ------------------------------
def chat(user_input):
    
    if safety_filter(user_input):
        return "⚠️ I cannot help with that. Please contact a healthcare professional immediately."
    
    prompt = SYSTEM_PROMPT + f"\nUser: {user_input}\nAssistant:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Remove the prompt from response
    response = response.replace(prompt, "").strip()
    
    return response

# ------------------------------
# 5️⃣ Chat Loop
# ------------------------------
print("\n🩺 FREE General Health Assistant Chatbot")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "exit":
        print("Chatbot: Stay healthy! 👋")
        break
    
    reply = chat(user_input)
    
    print("\nChatbot:", reply)
    print("\n----------------------------------\n")

Loading model... please wait (first time takes 1-2 minutes)


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

You:  I have a headache, what should I do?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Chatbot: A headache can be caused by many things, such as stress, dehydration, or a viral infection. To help relieve your headache, you can try taking over-the-counter pain relievers like acetaminophen or ibuprofen. You should also try to stay hydrated by drinking plenty of water. If your headache is severe, lasts for a long time, or is accompanied by other symptoms like dizziness or nausea, you should see a doctor. This information is for educational purposes only and not medical advice.

----------------------------------



You:  exit


Chatbot: Stay healthy! 👋
